# Загрузка и очистка данных TTRS

**Цель:** загрузить сырые транзакции клиентов Т-Банка (датасет TTRS), провести первичный осмотр, очистить данные от пропусков и дубликатов, привести типы и сохранить подготовленный датасет для дальнейшего анализа.

**Источник:** [TTRS: Tinkoff Transactions Recommender System benchmark](https://huggingface.co/datasets/pytorch-lifestream/ttrs) (CC BY-NC-SA 4.0).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Настройка отображения
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Пути к данным
RAW_PATH = Path('../data/raw/ttrs.csv')
PROCESSED_PATH = Path('../data/processed/transactions_clean.csv')

## 1. Загрузка сырых данных

Файл содержит ~14 млн транзакций. Загружаем его в DataFrame. 

In [2]:
df = pd.read_csv(RAW_PATH)

print(f"Размер датасета: {df.shape[0]:,} строк, {df.shape[1]} колонок")
df.head()

Размер датасета: 14,287,287 строк, 8 колонок


,party_rk,financial_account_type_cd,transaction_type_desc,merchant_type,merchant_group_rk,category,transaction_dttm,transaction_amt
0,1,1,1,1,0,1,2019-01-01,84
1,2,1,1,2,2,2,2019-01-01,410
2,3,2,1,3,0,3,2019-01-01,701
3,4,2,1,4,3,4,2019-01-01,6203
4,5,2,1,5,4,3,2019-01-01,734


## 2. Первичный осмотр

Смотрим на типы данных, пропуски и базовые статистики.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14287287 entries, 0 to 14287286
Data columns (total 8 columns):
 #   Column                     Dtype
---  ------                     -----
 0   party_rk                   int64
 1   financial_account_type_cd  int64
 2   transaction_type_desc      int64
 3   merchant_type              int64
 4   merchant_group_rk          int64
 5   category                   int64
 6   transaction_dttm           str  
 7   transaction_amt            int64
dtypes: int64(7), str(1)
memory usage: 1008.3 MB


In [4]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Пропуски': missing, 'Доля, %': missing_pct})
missing_df[missing_df['Пропуски'] > 0]

,Пропуски,"Доля, %"


**Вывод:** пропусков в данных нет — все 8 колонок заполнены полностью.

In [5]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
party_rk,14287287.00,NaN,NaN,NaN,19885.95,14727.02,1.00,7135.00,15978.00,33366.00,50000.00
financial_account_type_cd,14287287.00,NaN,NaN,NaN,1.68,0.47,1.00,1.00,2.00,2.00,2.00
transaction_type_desc,14287287.00,NaN,NaN,NaN,1.45,0.84,1.00,1.00,1.00,1.00,4.00
merchant_type,14287287.00,NaN,NaN,NaN,22.46,28.91,0.00,5.00,13.00,28.00,465.00
merchant_group_rk,14287287.00,NaN,NaN,NaN,87.89,269.92,0.00,0.00,0.00,45.00,2874.00
category,14287287.00,NaN,NaN,NaN,8.21,6.71,1.00,3.00,7.00,11.00,37.00
transaction_dttm,14287287,425,2019-11-12,92692,NaN,NaN,NaN,NaN,NaN,NaN,NaN
transaction_amt,14287287.00,NaN,NaN,NaN,3159.65,24292.27,0.00,159.00,388.00,1214.00,20000100.00


## 3. Очистка данных

На этом этапе:
- приводим `transaction_dttm` к типу `datetime`;
- проверяем и удаляем полные дубликаты;
- обрабатываем пропуски;
- фильтруем явные выбросы по сумме (отрицательные или нулевые транзакции).

In [6]:
# Приводим дату к datetime
df['transaction_dttm'] = pd.to_datetime(df['transaction_dttm'])

# Убедимся, что сумма — число с плавающей точкой
df['transaction_amt'] = df['transaction_amt'].astype(float)

df.dtypes

party_rk                              int64
financial_account_type_cd             int64
transaction_type_desc                 int64
merchant_type                         int64
merchant_group_rk                     int64
category                              int64
transaction_dttm             datetime64[us]
transaction_amt                     float64
dtype: object

In [7]:
# Проверяем, есть ли в датах время (отличное от 00:00:00)
unique_times = df['transaction_dttm'].dt.time.nunique()
print(f"Уникальных значений времени: {unique_times}")

if unique_times == 1:
    print("Время во всех транзакциях одинаковое (00:00:00). Признак 'hour' будет бесполезен.")

Уникальных значений времени: 1
Время во всех транзакциях одинаковое (00:00:00). Признак 'hour' будет бесполезен.


In [8]:
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f"Удалено дубликатов: {before - after:,}")

Удалено дубликатов: 238,855


In [9]:
df = df.dropna()
print(f"Осталось строк после удаления пропусков: {len(df):,}")

Осталось строк после удаления пропусков: 14,048,432


In [10]:
# Оставляем только положительные суммы
df = df[df['transaction_amt'] > 0]
print(f"Осталось строк после фильтрации по сумме: {len(df):,}")

Осталось строк после фильтрации по сумме: 14,048,427


**Вывод:** удалено 238 855 полных дубликатов (1.7% от исходного объёма), пропусков не было, отфильтровано 5 транзакций с нулевой суммой. Данные готовы к созданию признаков.

## 4. Создание новых признаков

Для дальнейшего анализа (EDA, RFM, кластеризация) добавим временные признаки:
- год, месяц, день;
- день недели;
- признак выходного дня.

In [11]:
df['year'] = df['transaction_dttm'].dt.year
df['month'] = df['transaction_dttm'].dt.month
df['day'] = df['transaction_dttm'].dt.day
df['day_of_week'] = df['transaction_dttm'].dt.dayofweek  # 0 = понедельник
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

# Признак 'hour' не создаём, так как время отсутствует
df[['transaction_dttm', 'year', 'month', 'day', 'day_of_week', 'is_weekend']].head()

,transaction_dttm,year,month,day,day_of_week,is_weekend
0,2019-01-01,2019,1,1,1,0
1,2019-01-01,2019,1,1,1,0
2,2019-01-01,2019,1,1,1,0
3,2019-01-01,2019,1,1,1,0
4,2019-01-01,2019,1,1,1,0


In [12]:
# Отфильтровываем верхний 0.1% сумм (выбросы)
threshold = df['transaction_amt'].quantile(0.999)
before = len(df)
df = df[df['transaction_amt'] <= threshold]
after = len(df)
print(f"Порог отсечения (99.9-й перцентиль): {threshold:,.0f} ₽")
print(f"Удалено выбросов: {before - after:,} строк")

Порог отсечения (99.9-й перцентиль): 200,100 ₽
Удалено выбросов: 12,090 строк


**Вывод:** отфильтрованы экстремальные суммы выше 99.9-го перцентиля. Это защитит RFM-метрики и кластеризацию от искажений.

## 5. Сохранение очищенного датасета

Сохраняем подготовленные данные для использования в следующих ноутбуках.

In [13]:
df.to_csv(PROCESSED_PATH, index=False)
print(f"Данные сохранены в {PROCESSED_PATH}")

Данные сохранены в ..\data\processed\transactions_clean.csv


In [14]:
print(f"Итоговый размер: {df.shape[0]:,} строк, {df.shape[1]} колонок")
print(f"Период данных: с {df['transaction_dttm'].min().date()} по {df['transaction_dttm'].max().date()}")
print(f"Уникальных пользователей: {df['party_rk'].nunique():,}")

Итоговый размер: 14,036,337 строк, 13 колонок
Период данных: с 2019-01-01 по 2020-02-29
Уникальных пользователей: 50,000


## 6. Итоговые выводы

- **Исходный объём:** 14 287 287 транзакций, 8 колонок.
- **После очистки:** 14 036 337 строк (удалено 238 855 дубликатов, 5 нулевых сумм, отфильтровано 12 090 выбросов выше 99.9-го перцентиля).
- **Период данных:** 2019-01-01 – 2020-02-29 (14 месяцев).
- **Уникальных пользователей:** 50 000.
- **Пропусков нет.**
- **Время в транзакциях отсутствует** — анализ по часам невозможен, признак `hour` не создавался.
- Данные сохранены в `data/processed/transactions_clean.csv` и готовы к EDA и RFM-анализу.